# TP 3 Transformer -> Stabiliser puis transformer chaque token

Paul Pitiot, ING 3 Option IA Groupe B

## Objectif retenu

Construire les deux briques qui agissent sur chaque token séparément : une `LayerNorm` écrite à la main (validée contre `nn.LayerNorm`) et le MLP `C → 4C → C`, puis les assembler avec le résiduel pré-norm. Étapes du sujet : 1 coder `LayerNorm`, 2 choisir l'axe, 3 construire le feed-forward, 4 et 5 ajouter le résiduel et tester des variantes.

`x (B, T, C) → LayerNorm (B, T, C) → MLP C → 4C → C (B, T, C) → x + Δx (B, T, C)`

Livrable : une `LayerNorm` validée contre `nn.LayerNorm`, un MLP `C → 4C → C` et un test du résiduel. Contrairement à l'attention, ces opérations ne mélangent pas les positions (vérifié par perturbation, partie 5). Aucun entraînement ; les graines sont fixées et aucun fichier n'est lu ni écrit.

## 0. Environnement et config

Constantes en majuscules ; graine 1337 comme dans les TP précédents. Les étapes 1 à 3 travaillent sur des vecteurs de dimension 100, l'étape 4 sur `(B, T, n_embd)` avec `n_embd = 64` (config du TP 4).

In [1]:
import platform

import torch
import torch.nn as nn
from torch.nn import functional as F

# ---- config ----
SEED = 1337                      # graine de toutes les étapes aléatoires
DIM_LN = 100                     # étapes 1 à 3 : dimension d'un vecteur (LayerNorm1d(100), FeedForward(100))
N_TOKENS_LN = 32                 # étapes 2 et 3 : nombre de tokens du tableau 2D (32, 100)
B, T = 4, 8                      # étapes 4 et 5 : B séquences indépendantes de T tokens
N_EMBD = 64                      # étapes 4 et 5 : dimension de l'embedding (C), comme dans le TP 4
FACTEURS_VARIANTE = (2, 4, 8)    # étape 5, variante : largeur cachée = facteur x C
ECHELLES_ANALYSE = (0.1, 1.0, 10.0, 100.0)   # partie 6 : échelles de tokens comparées
DIMS_ANALYSE = (16, 64, 256)     # partie 6 : valeurs de C comparées
N_GRAINES_ANALYSE = 20           # partie 6 : graines tirées par valeur de C

print(f"python {platform.python_version()} | torch {torch.__version__}")
print(f"SEED = {SEED} | DIM_LN = {DIM_LN} | N_TOKENS_LN = {N_TOKENS_LN}")
print(f"B = {B} | T = {T} | N_EMBD = {N_EMBD} | FACTEURS_VARIANTE = {FACTEURS_VARIANTE}")

python 3.11.15 | torch 2.14.0
SEED = 1337 | DIM_LN = 100 | N_TOKENS_LN = 32
B = 4 | T = 8 | N_EMBD = 64 | FACTEURS_VARIANTE = (2, 4, 8)


## 1. Étape 1 : coder LayerNorm

Pour chaque vecteur : moyenne et variance (biaisée) sur le dernier axe, on centre et on réduit, puis `γ` et `β` appris : `y = γ · (x - μ) / √(σ² + ε) + β`.

In [2]:
class LayerNorm1d:
    def __init__(self, dim, eps=1e-5):
        self.eps = eps
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

    def __call__(self, x):
        xmean = x.mean(dim=-1, keepdim=True)
        xvar = x.var(
            dim=-1, keepdim=True, unbiased=False
        )
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        return self.gamma * xhat + self.beta


# contrôles : formes, paramètres, et chaque token ressort centré (moyenne 0) et réduit (variance biaisée 1)
torch.manual_seed(SEED)
ln = LayerNorm1d(N_EMBD)
x = torch.randn(B, T, N_EMBD)
y = ln(x)

print(y.shape, ln.gamma.shape, ln.beta.shape)
print(y.mean(dim=-1).abs().max().item(), (y.var(dim=-1, unbiased=False) - 1).abs().max().item())

assert y.shape == x.shape == (B, T, N_EMBD)
assert ln.gamma.shape == ln.beta.shape == (N_EMBD,)
assert torch.allclose(y.mean(dim=-1), torch.zeros(B, T), atol=1e-6)
assert torch.allclose(y.var(dim=-1, unbiased=False), torch.ones(B, T), atol=1e-4)

torch.Size([4, 8, 64]) torch.Size([64]) torch.Size([64])
3.3527612686157227e-08 1.6808509826660156e-05


**Questions**

- Sur quel axe normaliser ? À quoi servent `ε`, `γ` et `β` ?

Réponses :

- On normalise sur l'axe des features d'un token (`dim=-1`) : chaque vecteur utilise ses propres statistiques.
- `ε` évite la division par zéro. `γ` et `β` redonnent à chaque feature une échelle et un décalage appris ; à l'initialisation, `γ = 1` et `β = 0`.

> *Commentaire - 1 :* sur `(4, 8, 64)`, la classe garde la forme, et `γ` et `β` ont la forme `(64,)`. Chaque token sort centré (moyenne au plus 3,4e-8) et de variance biaisée égale à 1 à 1,7e-5 près, écart dû à `ε`.

## 2. Étape 2 : choisir l'axe de normalisation

`x[0, :]` est le token 0 à travers ses 100 features, `x[:, 0]` la feature 0 à travers les 32 tokens. Prédiction avant exécution : `x[0, :]` donnera environ `(0, 1)`, `x[:, 0]` n'est pas contraint.

In [3]:
torch.manual_seed(SEED)
module = LayerNorm1d(DIM_LN)

x_brut = torch.randn(N_TOKENS_LN, DIM_LN)
x = module(x_brut)

print(x[:, 0].mean(), x[:, 0].std())
print(x[0, :].mean(), x[0, :].std())

# validation du sujet : comparaison avec nn.LayerNorm
ref = nn.LayerNorm(DIM_LN)
print(torch.allclose(module(x_brut), ref(x_brut), atol=1e-6))

assert torch.allclose(module(x_brut), ref(x_brut), atol=1e-6)
assert x[0, :].mean().abs() < 1e-6
assert torch.isclose(x[0, :].std(), torch.tensor((DIM_LN / (DIM_LN - 1)) ** 0.5), atol=1e-4)
assert not torch.isclose(x[:, 0].std(), torch.tensor(1.0), atol=1e-2)   # la feature 0 n'est pas normalisée

tensor(0.1476) tensor(0.8847)
tensor(-2.3842e-09) tensor(1.0050)
True


Variantes : (a) axe des tokens, (b) `γ` et `β` non triviaux, (c) `ε` sur un vecteur constant, (d) invariance à `a·x + b`, (e) indépendance entre tokens.

In [4]:
# (a) mauvais axe : moyenne et variance prises à travers les 32 tokens (dim=0)
x_col = (x_brut - x_brut.mean(dim=0, keepdim=True)) / torch.sqrt(
    x_brut.var(dim=0, keepdim=True, unbiased=False) + module.eps
)
print("(a) axe tokens   x[:, 0] :", x_col[:, 0].mean().item(), x_col[:, 0].std().item())
print("(a) axe tokens   x[0, :] :", x_col[0, :].mean().item(), x_col[0, :].std().item())

# (b) gamma et beta non triviaux, recopiés dans nn.LayerNorm
torch.manual_seed(SEED)
module_gb = LayerNorm1d(DIM_LN)
module_gb.gamma = torch.rand(DIM_LN) + 0.5
module_gb.beta = torch.randn(DIM_LN)
ref_gb = nn.LayerNorm(DIM_LN)
with torch.no_grad():
    ref_gb.weight.copy_(module_gb.gamma)
    ref_gb.bias.copy_(module_gb.beta)
assert torch.allclose(module_gb(x_brut), ref_gb(x_brut), atol=1e-6)
y_gb = module_gb(x_brut)
print("(b) avec gamma, beta  x[0, :] :", y_gb[0].mean().item(), y_gb[0].std().item())

# (c) epsilon sur un vecteur constant (variance nulle)
v = torch.full((1, DIM_LN), 3.0)
print("(c) eps = 1e-5 :", module(v)[0, :3])
print("(c) eps = 0    :", LayerNorm1d(DIM_LN, eps=0.0)(v)[0, :3])

# (d) invariance d'un token à x -> a * x + b (a > 0)
assert torch.allclose(module(3 * x_brut + 5), x, atol=1e-4)

# (e) pas de mélange entre tokens : changer le token 5 ne modifie aucune autre ligne
x_modif = x_brut.clone()
torch.manual_seed(SEED + 1)
x_modif[5] = torch.randn(DIM_LN)
autres = [i for i in range(N_TOKENS_LN) if i != 5]
assert torch.equal(module(x_modif)[autres], x[autres])
print("(d) invariance a*x+b et (e) indépendance des tokens : vérifiées")

(a) axe tokens   x[:, 0] : 7.450580596923828e-09 1.0159944295883179
(a) axe tokens   x[0, :] : 0.04171385616064072 1.059760570526123
(b) avec gamma, beta  x[0, :] : -0.13661475479602814 1.5592483282089233
(c) eps = 1e-5 : tensor([0., 0., 0.])
(c) eps = 0    : tensor([nan, nan, nan])
(d) invariance a*x+b et (e) indépendance des tokens : vérifiées


**Questions**

- Prédisez quelle ligne donnera environ `(0, 1)` avant d'exécuter le code.

Réponse : `x[0, :]` donnera environ `(0, 1)`. La couche normalise les 100 features de chaque token, pas les 32 valeurs d'une feature.

> *Commentaire - 2 :* la prédiction est confirmée. `x[0, :]` donne `(-2,4e-9 ; 1,0050)` (`.std()` est non biaisé, d'où `√(100/99)`) et `x[:, 0]` donne `(0,1476 ; 0,8847)`. `torch.allclose` avec `nn.LayerNorm(100)` vaut `True`.
>
> Variantes : (a) avec l'axe des tokens, les rôles s'échangent ; (b) avec `γ` et `β` non triviaux, le résultat reste égal à `nn.LayerNorm` ; (c) sur un vecteur constant, `ε = 1e-5` donne 0 et `ε = 0` donne `nan` ; (d) et (e) sont vérifiées par `assert`.

## 3. Étape 3 : construire le réseau feed-forward

`C → 4C`, ReLU, `4C → C` : le MLP augmente temporairement la dimension puis revient à celle de l'embedding.

In [5]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
        )

    def forward(self, x):
        return self.net(x)


torch.manual_seed(SEED)
ffwd = FeedForward(DIM_LN)
x_ffn = torch.randn(N_TOKENS_LN, DIM_LN)
delta = ffwd(x_ffn)
print(delta.shape)

# formes intermédiaires : (32, 100) -> (32, 400) -> (32, 400) -> (32, 100)
pre = ffwd.net[0](x_ffn)
act = ffwd.net[1](pre)
print(pre.shape, act.shape)
assert pre.shape == act.shape == (N_TOKENS_LN, 4 * DIM_LN)
assert delta.shape == x_ffn.shape

# proportion d'activations positives (neurones actifs) et nombre de paramètres
prop_positive = (act > 0).float().mean().item()
n_params = sum(p.numel() for p in ffwd.parameters())
print(f"activations positives : {prop_positive:.3f} | paramètres : {n_params}")
assert n_params == 2 * 4 * DIM_LN**2 + 5 * DIM_LN       # W_up + W_down + biais

# le MLP s'applique à chaque token séparément : (B, T, C) donne la même chose que token par token
torch.manual_seed(SEED)
xs = torch.randn(B, T, DIM_LN)
ys = ffwd(xs)
assert ys.shape == xs.shape
for ib, t in ((0, 0), (1, 3), (3, 7)):
    assert torch.allclose(ys[ib, t], ffwd(xs[ib, t]), atol=1e-6)

torch.Size([32, 100])
torch.Size([32, 400]) torch.Size([32, 400])
activations positives : 0.507 | paramètres : 80500


**Questions**

- Quelle proportion d'activations est positive ? Pourquoi revenir à `C` ?

Réponses :

- Environ la moitié des activations est positive (0,507 ici), parce que les poids sont symétriques autour de 0 et que l'entrée est centrée.
- On revient à `C` pour pouvoir ajouter la sortie à l'entrée (`x + Δx`) et empiler les blocs.

> *Commentaire - 3 :* les formes sont `(32, 100) → (32, 400) → (32, 400) → (32, 100)` et le MLP compte 80 500 paramètres (`2·4·C² + 5·C`, vérifié). Sur `(B, T, 100)`, la sortie du token `(b, t)` est la même que celle du vecteur seul : le MLP agit token par token.

## 4. Étapes 4 et 5 : ajouter le résiduel (pré-norm)

`x_out = x + MLP(LayerNorm(x))` sur un `(B, T, C)` avec `C = 64`.

In [6]:
torch.manual_seed(SEED)
n_embd = N_EMBD
layer_norm = nn.LayerNorm(n_embd)
ffwd = FeedForward(n_embd)

x = torch.randn(B, T, n_embd)
delta = ffwd(layer_norm(x))
x_out = x + delta

print(x.shape)
print(delta.shape)
print(x_out.shape)

# contrôles : formes, et le résiduel ajoute sans remplacer (x_out - x = delta)
assert x.shape == delta.shape == x_out.shape == (B, T, n_embd)
assert torch.allclose(x_out - x, delta, atol=1e-6)
print(f"|delta| / |x| = {(delta.norm() / x.norm()).item():.3f}")

torch.Size([4, 8, 64])
torch.Size([4, 8, 64])
torch.Size([4, 8, 64])
|delta| / |x| = 0.231


Variantes du sujet : 1. `GELU` à la place de `ReLU`, 2. `C → 2C → C` puis `C → 8C → C`, 3. nombre de paramètres, 4. sans addition résiduelle (comparée par cosinus avec l'entrée, car les formes seules ne distinguent pas les deux cas).

In [7]:
class FeedForwardVar(nn.Module):
    # même MLP que FeedForward, avec facteur de largeur et activation en paramètres
    def __init__(self, n_embd, facteur=4, activation=nn.ReLU):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, facteur * n_embd),
            activation(),
            nn.Linear(facteur * n_embd, n_embd),
        )

    def forward(self, x):
        return self.net(x)


def n_params(module):
    return sum(p.numel() for p in module.parameters())


# 1) ReLU contre GELU, même graine donc mêmes poids : seule l'activation change
torch.manual_seed(SEED)
mlp_relu = FeedForwardVar(n_embd, 4, nn.ReLU)
torch.manual_seed(SEED)
mlp_gelu = FeedForwardVar(n_embd, 4, nn.GELU)
h = layer_norm(x)
hid_relu = mlp_relu.net[1](mlp_relu.net[0](h))
hid_gelu = mlp_gelu.net[1](mlp_gelu.net[0](h))
print(f"ReLU : {(hid_relu == 0).float().mean().item():.3f} d'activations exactement nulles | GELU : {(hid_gelu == 0).float().mean().item():.3f}")
print(f"écart moyen des sorties ReLU / GELU : {(mlp_relu(h) - mlp_gelu(h)).abs().mean().item():.4f} (|sortie ReLU| moyenne : {mlp_relu(h).abs().mean().item():.4f})")
assert (x + mlp_gelu(h)).shape == x_out.shape
assert n_params(mlp_relu) == n_params(mlp_gelu)

# 2) et 3) largeur cachée : formes et nombre de paramètres
print("facteur  largeur  sortie        paramètres")
for f in FACTEURS_VARIANTE:
    torch.manual_seed(SEED)
    mlp = FeedForwardVar(n_embd, f)
    sortie = x + mlp(layer_norm(x))
    print(f"{f:<8} {f * n_embd:<8} {tuple(sortie.shape)}  {n_params(mlp):>10}")
    assert sortie.shape == (B, T, n_embd)
    assert n_params(mlp) == 2 * f * n_embd**2 + (f + 1) * n_embd

# 4) sans addition résiduelle : même forme, mais l'entrée n'est plus dans la sortie
cos = lambda a, b: F.cosine_similarity(a, b, dim=-1).mean().item()
x_sans = delta                      # sortie du bloc sans résiduel
print(f"formes : avec {tuple(x_out.shape)} | sans {tuple(x_sans.shape)}")
print(f"cosinus moyen entre la sortie et l'entrée x : avec résiduel {cos(x_out, x):.3f} | sans {cos(x_sans, x):.3f}")

# si le MLP ne revenait pas à C, l'addition serait impossible
sans_retour = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.ReLU())
try:
    x + sans_retour(layer_norm(x))
except RuntimeError as e:
    print("sans la couche 4C -> C :", str(e).splitlines()[0])

ReLU : 0.499 d'activations exactement nulles | GELU : 0.000
écart moyen des sorties ReLU / GELU : 0.0566 (|sortie ReLU| moyenne : 0.1846)
facteur  largeur  sortie        paramètres
2        128      (4, 8, 64)       16576
4        256      (4, 8, 64)       33088
8        512      (4, 8, 64)       66112
formes : avec (4, 8, 64) | sans (4, 8, 64)
cosinus moyen entre la sortie et l'entrée x : avec résiduel 0.974 | sans -0.003
sans la couche 4C -> C : The size of tensor a (64) must match the size of tensor b (256) at non-singleton dimension 2


> *Commentaire - 4 :*
> - Les formes de `x`, `delta` et `x_out` sont `(4, 8, 64)`, et `x_out - x` égale `delta`.
> - Avec `GELU`, aucune activation n'est exactement nulle, contre 49,9 % avec `ReLU`, pour le même nombre de paramètres.
> - Le MLP compte 16 576, 33 088 et 66 112 paramètres pour `2C`, `4C` et `8C` (`2·f·C² + (f+1)·C`).
> - Sans résiduel, les formes sont identiques mais le cosinus avec l'entrée passe de 0,974 à -0,003. Un MLP qui ne revient pas à `C` fait échouer l'addition.

## 5. Bilan du TP 3

Test de perturbation sur le bloc de la partie 4.

In [8]:
# LayerNorm conserve (B, T, C)
assert layer_norm(x).shape == x.shape

# le bloc ne mélange pas les tokens : on change le token 5 de la séquence 0, les 7 autres sorties de cette séquence
# et toutes les sorties des autres séquences ne bougent pas
x_modif = x.clone()
torch.manual_seed(SEED + 1)
x_modif[0, 5] = torch.randn(n_embd)
x_out_modif = x_modif + ffwd(layer_norm(x_modif))

masque = torch.ones(B, T, dtype=torch.bool)
masque[0, 5] = False
print("autres tokens inchangés :", torch.equal(x_out[masque], x_out_modif[masque]),
      "| token modifié changé :", not torch.equal(x_out[0, 5], x_out_modif[0, 5]))
assert torch.equal(x_out[masque], x_out_modif[masque])
assert not torch.equal(x_out[0, 5], x_out_modif[0, 5])

# l'attention du TP 2, elle, mélange : mêmes entrées, un seul token change, les sorties des positions suivantes bougent
torch.manual_seed(SEED)
wq, wk, wv = (nn.Linear(n_embd, n_embd, bias=False) for _ in range(3))
def attention(z):
    sc = wq(z) @ wk(z).transpose(-2, -1) * n_embd**-0.5
    sc = sc.masked_fill(torch.tril(torch.ones(T, T)) == 0, float('-inf'))
    return F.softmax(sc, dim=-1) @ wv(z)
att_a, att_b = attention(x), attention(x_modif)
print("attention : positions 6 et 7 de la séquence 0 inchangées :", torch.allclose(att_a[0, 6:], att_b[0, 6:]))
assert not torch.allclose(att_a[0, 6:], att_b[0, 6:])
print("OK : LayerNorm conserve (B, T, C), le MLP et le bloc ne mélangent pas les tokens, le résiduel ajoute Δx à x.")

autres tokens inchangés : True | token modifié changé : True
attention : positions 6 et 7 de la séquence 0 inchangées : False
OK : LayerNorm conserve (B, T, C), le MLP et le bloc ne mélangent pas les tokens, le résiduel ajoute Δx à x.


**Questions**

- Quelle dimension `LayerNorm` conserve-t-elle ? Le MLP mélange-t-il les tokens ? À quoi sert la connexion résiduelle ?

Réponses :

- `LayerNorm` conserve toutes les dimensions : la forme `(B, T, C)` ne change pas, seules les valeurs changent.
- Le MLP ne mélange pas les tokens : modifier un token ne change aucune autre sortie. Seule l'attention fait circuler l'information entre les positions.
- La connexion résiduelle ajoute une correction à `x` au lieu de le remplacer (`E' = E + ΔE`) et permet d'empiler les blocs.

> *Commentaire - 5 :* en changeant le token 5 de la séquence 0, les 31 autres sorties restent strictement identiques (`torch.equal`). Avec les mêmes entrées, l'attention du TP 2 fait bouger les positions 6 et 7 de cette séquence.

## 6. Analyse

Quatre vérifications : reproductibilité, échelle, activations, amplitude de `Δx`.

In [9]:
def bloc(seed, n=N_EMBD, facteur=4):
    # x, LayerNorm, MLP tirés dans le même ordre que la partie 4
    torch.manual_seed(seed)
    ln_ = nn.LayerNorm(n)
    mlp = FeedForwardVar(n, facteur)
    x_ = torch.randn(B, T, n)
    return x_, ln_, mlp


# 1) reproductibilité
def sortie(seed):
    # même ordre de tirage que la partie 4 : layer_norm, ffwd, puis x
    torch.manual_seed(seed)
    ln_ = nn.LayerNorm(N_EMBD)
    mlp = FeedForward(N_EMBD)
    x_ = torch.randn(B, T, N_EMBD)
    return x_ + mlp(ln_(x_))

assert torch.equal(sortie(SEED), sortie(SEED))                  # même graine : identique
assert torch.allclose(sortie(SEED), x_out, atol=1e-6)           # = partie 4
assert not torch.allclose(sortie(SEED + 1), x_out)              # autre graine : autre sortie
print(f"même graine : écart max {(sortie(SEED) - sortie(SEED)).abs().max().item():.1e} | graine + 1 : écart max {(sortie(SEED) - sortie(SEED + 1)).abs().max().item():.3f}")

même graine : écart max 0.0e+00 | graine + 1 : écart max 5.328


In [10]:
# 2) échelle : 1000 tokens dont l'échelle est fixée par groupe de 250, normalisés par la LayerNorm (axe features)
torch.manual_seed(SEED)
n_par_echelle = 250
base = torch.randn(len(ECHELLES_ANALYSE) * n_par_echelle, DIM_LN)
echelles = torch.tensor(ECHELLES_ANALYSE).repeat_interleave(n_par_echelle).unsqueeze(-1)
tokens = base * echelles + 7.0          # échelle différente par groupe, et un décalage commun
sortie_ln = LayerNorm1d(DIM_LN)(tokens)

print("échelle   std brut (moyen)   std après LayerNorm (moyen)   moyenne max |.| après")
for i, e in enumerate(ECHELLES_ANALYSE):
    g = slice(i * n_par_echelle, (i + 1) * n_par_echelle)
    print(f"{e:<9} {tokens[g].std(dim=-1).mean().item():>14.3f} {sortie_ln[g].std(dim=-1).mean().item():>24.4f} {sortie_ln[g].mean(dim=-1).abs().max().item():>22.1e}")
assert torch.allclose(sortie_ln.mean(dim=-1), torch.zeros(len(tokens)), atol=1e-4)

# même chose avec l'axe faux (tokens) : les statistiques d'un token dépendent alors des autres tokens du lot
faux = (tokens - tokens.mean(dim=0, keepdim=True)) / (tokens.std(dim=0, keepdim=True) + 1e-5)
print("axe faux, std moyen par token et par groupe :",
      [round(faux[i * n_par_echelle:(i + 1) * n_par_echelle].std(dim=-1).mean().item(), 3) for i in range(len(ECHELLES_ANALYSE))])

échelle   std brut (moyen)   std après LayerNorm (moyen)   moyenne max |.| après
0.1                0.100                   1.0045                1.1e-05
1.0                0.989                   1.0050                9.9e-07
10.0               9.967                   1.0050                1.2e-07
100.0             99.541                   1.0050                4.8e-08
axe faux, std moyen par token et par groupe : [0.032, 0.038, 0.201, 1.983]


In [11]:
# 3) proportion de neurones actifs à l'initialisation (entrée gaussienne), selon C et la graine
print("C     actifs : moyenne  écart-type  min    max   (sur %d graines)" % N_GRAINES_ANALYSE)
for c in DIMS_ANALYSE:
    props = []
    for s in range(N_GRAINES_ANALYSE):
        torch.manual_seed(SEED + s)
        mlp = FeedForward(c)
        z = torch.randn(256, c)
        props.append((mlp.net[0](z) > 0).float().mean().item())
    p = torch.tensor(props)
    print(f"{c:<5} {p.mean().item():>16.3f} {p.std().item():>10.3f} {p.min().item():>6.3f} {p.max().item():>6.3f}")

C     actifs : moyenne  écart-type  min    max   (sur 20 graines)
16               0.498      0.013  0.477  0.525
64               0.500      0.004  0.492  0.507
256              0.500      0.001  0.498  0.502


In [12]:
# 4) amplitude de delta = MLP(LayerNorm(x)) par rapport à x, selon la largeur cachée (plusieurs graines)
print("facteur  |delta| / |x| (moyenne sur %d graines)" % N_GRAINES_ANALYSE)
for f in FACTEURS_VARIANTE:
    rapports = []
    for s in range(N_GRAINES_ANALYSE):
        x_, ln_, mlp = bloc(SEED + s, facteur=f)
        rapports.append((mlp(ln_(x_)).norm() / x_.norm()).item())
    r = torch.tensor(rapports)
    print(f"{f:<8} {r.mean().item():.3f} (écart-type {r.std().item():.3f})")

facteur  |delta| / |x| (moyenne sur 20 graines)
2        0.239 (écart-type 0.012)
4        0.241 (écart-type 0.009)
8        0.239 (écart-type 0.011)


> *Commentaire - 6 :*
> - Reproductibilité : avec la même graine, la sortie est identique (écart 0) et égale à celle de la partie 4 ; avec la graine + 1, elle change.
> - Échelle : des tokens d'échelle 0,1 à 100 ressortent à écart-type 1,005 et moyenne nulle. Avec l'axe des tokens, l'écart-type reste proportionnel à l'échelle (0,032 à 1,983).
> - Activations : la proportion vaut 0,498 pour `C = 16`, puis 0,500 pour `C = 64` et `C = 256`, sur 20 graines. L'écart entre graines se resserre quand `C` grandit.
> - Amplitude de `Δx` : `|Δx| / |x|` vaut environ 0,24 pour `f = 2, 4, 8`, donc élargir le MLP ne la change pas à l'initialisation. L'initialisation de `nn.Linear` l'explique probablement, mais je ne l'ai pas testé.
>
> Limites : les poids ne sont pas entraînés, les entrées sont gaussiennes et il n'y a qu'un bloc.

## 7. Consignes du sujet

| Consigne | Où |
|---|---|
| Étape 1 : `LayerNorm1d` (moyenne, variance, `ε`, `γ`, `β`), question sur l'axe | partie 1 |
| Étape 2 : `x[:, 0]` contre `x[0, :]`, prédiction avant exécution, `torch.allclose` avec `nn.LayerNorm(100)` | partie 2 |
| Étape 3 : `FeedForward` `C → 4C → C`, forme de `delta`, proportion d'activations positives | partie 3 (mesure répétée en partie 6) |
| Étapes 4 et 5 : `x + FeedForward(LayerNorm(x))`, formes de `x`, `delta`, `x_out` | partie 4 |
| Variantes : `GELU`, `2C` et `8C`, nombre de paramètres, sans résiduel | partie 4 |
| Bilan : dimension conservée, le MLP ne mélange pas les tokens, rôle du résiduel | partie 5 |
| Formes vérifiées, rôle expliqué, variante testée à chaque étape | `assert` et variante dans chaque partie |